# 02 — Test risk-engine (Redis outputs après seed surface)

Smoke test du container `fxvol-risk` — étape 2/5. Valide que **les sorties Redis du cycle 2s sont bien produites** : heartbeat ISO, `latest_greeks:portfolio` JSON avec les Greeks attendus, et que les Greeks reflètent bien le spot lu depuis Redis.

**Setup particulier — seed manuel** : le cycle risk-engine skip silencieusement si `latest_spot:EURUSD` OU `latest_vol_surface:EURUSD` est absent (cf. `engine.py:_read_spot/_read_surface`). Comme **vol-engine** met ~5 min à publier sa première surface (cycle 30s + GARCH calc), on seed **manuellement une surface factice** au début de ce notebook pour pouvoir valider risk-engine indépendamment de l'état vol-engine.

Le spot vient soit de market-data (qui publie `str(mid)` plain float, accepté par le patch R9 sandbox), soit de notre seed (dict JSON `{mid, bid, ask}`) — les 2 formats marchent.

**Couvre** :
1. Seed `latest_vol_surface:EURUSD` JSON valide + wait 5s (3 cycles risk-engine de 2s)
2. `heartbeat:risk_engine` frais (ISO-8601, age < 5s, TTL > 0)
3. `latest_greeks:portfolio` JSON valide avec schéma `{timestamp, greeks:{delta, gamma, vega, theta, spot}}`
4. Tous les Greeks (delta, gamma, vega, theta) sont des floats finis numériques
5. **Cohérence spot** : la valeur `greeks.spot` matche le `latest_spot:EURUSD` (preuve que l'engine a bien lu Redis)
6. `latest_pnl_curve` est None/absent (attendu avec positions stub vides — confirmé par `engine.py:108` qui fait `pnl_curve = ... if positions else None`)

**Préreq** :
- Notebook 01 vert (container UP + healthy + env vars)
- market-data running (publie `latest_spot:EURUSD`) — sinon on overwrite avec un seed factice
- Redis exposé sur `localhost:6380`

## Setup + seed surface

In [1]:
import json
import math
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
SYMBOL = "EURUSD"
EXPECTED_GREEKS_KEYS = {"delta", "gamma", "vega", "theta", "spot"}

# Une cycle risk-engine = 2s. Avec 5s d'attente après le seed, on a
# au moins 2 cycles pour que la première publication apparaisse.
WAIT_AFTER_SEED_S = 5.0

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL — check 'docker compose ps'")
print(f"Connected -> {REDIS_URL}")

Connected -> redis://localhost:6380/0


## 1. Seed `latest_vol_surface:EURUSD` + wait

Format du surface attendu par `engine.py:_read_surface` : wrapper `{"surface": <dict tenor → label → {iv, strike}>}`. Format minimal qui permet à `_aggregate_greeks` de tomber sur `FALLBACK_IV` si on n'avait pas de positions, mais avec stub vide il s'en fout — on remplit par cohérence de schéma.

In [2]:
fake_surface = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "surface": {
        "1M": {
            "atm": {"iv": 0.07, "strike": 1.17},
            "25dc": {"iv": 0.072, "strike": 1.18},
            "25dp": {"iv": 0.075, "strike": 1.16},
        },
        "3M": {
            "atm": {"iv": 0.078, "strike": 1.17},
        },
    },
}

r.set(f"latest_vol_surface:{SYMBOL}", json.dumps(fake_surface), ex=600)
record("seed latest_vol_surface", True,
       f"surface.tenors = {list(fake_surface['surface'].keys())}")

print(f"  [INFO] waiting {WAIT_AFTER_SEED_S}s pour ≥ 2 cycles risk-engine...")
time.sleep(WAIT_AFTER_SEED_S)

  [OK  ] seed latest_vol_surface  | surface.tenors = ['1M', '3M']
  [INFO] waiting 5.0s pour ≥ 2 cycles risk-engine...


## 2. Heartbeat frais + ISO-8601 + TTL

**Ce que tu dois voir** : timestamp ISO, age < 5s (cycle 2s, on a attendu 5s donc le dernier heartbeat est ≤ 2s d'âge), TTL > 0.

In [3]:
print("== 2. heartbeat ==")

hb_raw = r.get("heartbeat:risk_engine")
record("heartbeat:risk_engine présent",
       hb_raw is not None,
       hb_raw if hb_raw else "<missing — cycle skip ? check docker logs fxvol-risk>")

if hb_raw:
    try:
        hb_dt = datetime.fromisoformat(hb_raw.replace("Z", "+00:00"))
        age_s = (datetime.now(timezone.utc) - hb_dt).total_seconds()
        record("heartbeat ISO-8601 parsable", True, f"parsed = {hb_dt.isoformat()}")
        record("heartbeat age < 5s", age_s < 5.0, f"age = {age_s:.2f}s")
    except ValueError as e:
        record("heartbeat ISO-8601 parsable", False, f"raw={hb_raw!r}: {e}")
    ttl = r.ttl("heartbeat:risk_engine")
    record("heartbeat TTL > 0", ttl > 0, f"TTL = {ttl}s")

== 2. heartbeat ==
  [OK  ] heartbeat:risk_engine présent  | 2026-04-28T15:20:49.384051Z
  [OK  ] heartbeat ISO-8601 parsable  | parsed = 2026-04-28T15:20:49.384051+00:00
  [OK  ] heartbeat age < 5s  | age = 0.01s
  [OK  ] heartbeat TTL > 0  | TTL = 30s


## 3. `latest_greeks:portfolio` — schéma + types

**Schéma attendu** (cf. `bus/publisher.py:publish_risk_update`) :
```json
{
  "timestamp": "ISO-8601 UTC",
  "greeks": {
    "delta": 0.0,
    "gamma": 0.0,
    "vega": 0.0,
    "theta": 0.0,
    "spot": 1.17047
  }
}
```
Les Greeks sont à 0 parce que `_positions_stub()` retourne `[]`. Ça reste un schéma valide ; ce qu'on teste ici c'est que **le pipeline produit le bon shape**, pas que les valeurs sont financièrement utiles (ça viendra quand on aura un vrai fetch positions).


In [4]:
print("== 3. latest_greeks:portfolio ==")

greeks_raw = r.get("latest_greeks:portfolio")
record("latest_greeks:portfolio présent",
       greeks_raw is not None,
       greeks_raw[:80] + "..." if greeks_raw and len(greeks_raw) > 80 else (greeks_raw or "<missing>"))

greeks_payload = None
if greeks_raw:
    try:
        greeks_payload = json.loads(greeks_raw)
        record("latest_greeks JSON valide", True, f"keys = {sorted(greeks_payload.keys())}")
    except json.JSONDecodeError as e:
        record("latest_greeks JSON valide", False, str(e))

if greeks_payload:
    has_top = {"timestamp", "greeks"} <= set(greeks_payload.keys())
    record("top-level keys = {timestamp, greeks}", has_top,
           f"actual = {sorted(greeks_payload.keys())}")
    g = greeks_payload.get("greeks", {})
    has_greeks = EXPECTED_GREEKS_KEYS <= set(g.keys())
    record(f"greeks keys = {sorted(EXPECTED_GREEKS_KEYS)}", has_greeks,
           f"actual = {sorted(g.keys())}")

== 3. latest_greeks:portfolio ==
  [OK  ] latest_greeks:portfolio présent  | {"timestamp": "2026-04-28T15:20:49.383543Z", "greeks": {"delta": 0.0, "gamma": 0...
  [OK  ] latest_greeks JSON valide  | keys = ['greeks', 'timestamp']
  [OK  ] top-level keys = {timestamp, greeks}  | actual = ['greeks', 'timestamp']
  [OK  ] greeks keys = ['delta', 'gamma', 'spot', 'theta', 'vega']  | actual = ['delta', 'gamma', 'spot', 'theta', 'vega']


## 4. Tous les Greeks sont des floats finis

Avec positions stub vides, on s'attend à `delta=gamma=vega=theta=0.0`, et `spot` à la valeur réelle lue depuis `latest_spot:EURUSD`. Tous doivent être des floats finis (pas de NaN, pas d'inf, pas de None).

Si jamais on voit un NaN sur les Greeks à 0, c'est un bug de calcul (ex: division par zéro implicite quelque part).

In [5]:
print("== 4. Greeks finite ==")

if not greeks_payload:
    record("Greeks finite", False, "skip (cf. §3)")
else:
    g = greeks_payload.get("greeks", {})
    for key in ("delta", "gamma", "vega", "theta", "spot"):
        val = g.get(key)
        is_ok = isinstance(val, (int, float)) and not (isinstance(val, float) and math.isnan(val))
        record(f"greeks.{key} = float fini", is_ok, f"value = {val}")

== 4. Greeks finite ==
  [OK  ] greeks.delta = float fini  | value = 0.0
  [OK  ] greeks.gamma = float fini  | value = 0.0
  [OK  ] greeks.vega = float fini  | value = 0.0
  [OK  ] greeks.theta = float fini  | value = 0.0
  [OK  ] greeks.spot = float fini  | value = 1.17077


## 5. Cohérence spot — `greeks.spot` ≈ `latest_spot:EURUSD`

**Le test clé** : la valeur `greeks.spot` que risk-engine a écrite doit matcher le `latest_spot:EURUSD` qu'il a lu (à un cycle près de delta — le spot peut bouger entre la lecture par risk-engine et notre lecture).

Tolérance : 0.5% (le spot peut bouger un peu en 5s sur un marché live). Si le delta dépasse ça → soit le pipeline est cassé, soit le marché a fait un mouvement fou.

**Pourquoi ce test compte** : c'est ça qui prouve que **le patch `_read_spot` (R9 sandbox)** marche. Avant patch, l'engine lisait None pour le spot et skippait → on n'aurait pas de greeks du tout. Maintenant on a un spot lu depuis Redis et reflété dans le payload greeks.

In [6]:
print("== 5. cohérence greeks.spot vs latest_spot ==")

spot_raw = r.get(f"latest_spot:{SYMBOL}")
if not spot_raw or not greeks_payload:
    record("cohérence spot", False, "skip (manque spot ou greeks)")
else:
    # latest_spot peut être plain float (market-data) OU JSON dict (notre seed).
    try:
        spot_parsed = json.loads(spot_raw)
        if isinstance(spot_parsed, dict):
            spot_redis = float(spot_parsed.get("mid") or spot_parsed.get("bid"))
        else:
            spot_redis = float(spot_parsed)
    except (ValueError, TypeError):
        spot_redis = None

    spot_engine = greeks_payload["greeks"].get("spot")
    if spot_redis is None or spot_engine is None:
        record("cohérence spot", False,
               f"spot_redis={spot_redis}, spot_engine={spot_engine}")
    else:
        delta_pct = abs(spot_engine - spot_redis) / spot_redis * 100
        record("|greeks.spot - latest_spot| / spot < 0.5%",
               delta_pct < 0.5,
               f"engine={spot_engine}, redis={spot_redis}, delta={delta_pct:.4f}%")

== 5. cohérence greeks.spot vs latest_spot ==
  [OK  ] |greeks.spot - latest_spot| / spot < 0.5%  | engine=1.17077, redis=1.17077, delta=0.0000%


## 6. `latest_pnl_curve` absent (attendu avec stub vide)

Comportement actuel de l'engine (`engine.py:108`) :
```python
pnl_curve = self._compute_pnl_curve(positions, F, surface) if positions else None
```

Avec `_positions_stub()` = `[]`, `pnl_curve` reste `None`, et `publisher.publish_risk_update` ne SET pas la clé `latest_pnl_curve`. Donc on s'attend à ce qu'elle soit **absente** ou **expirée**.

Quand on aura un vrai fetch positions (PR future), le test deviendra : `latest_pnl_curve` JSON `{timestamp, curve:{spots: [120 floats], pnls: [120 floats], spot}}`. Pour l'instant on documente que c'est un placeholder.

In [7]:
print("== 6. latest_pnl_curve (avec positions=[]) ==")

pnl_raw = r.get("latest_pnl_curve")
record("latest_pnl_curve absent (positions stub vides)",
       pnl_raw is None,
       "<absent>" if pnl_raw is None
       else f"présent (inattendu sauf si positions stub modifié) : {pnl_raw[:80]}")

== 6. latest_pnl_curve (avec positions=[]) ==
  [OK  ] latest_pnl_curve absent (positions stub vides)  | <absent>


## Récap final

In [8]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — Redis outputs sains. Pass au notebook 03 (pub/sub channel risk_update).")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
seed latest_vol_surface                                 OK      surface.tenors = ['1M', '3M']
heartbeat:risk_engine présent                           OK      2026-04-28T15:20:49.384051Z
heartbeat ISO-8601 parsable                             OK      parsed = 2026-04-28T15:20:49.384051+00:00
heartbeat age < 5s                                      OK      age = 0.01s
heartbeat TTL > 0                                       OK      TTL = 30s
latest_greeks:portfolio présent                         OK      {"timestamp": "2026-04-28T15:20:49.383543Z", "greeks": {"delta": 0.0, "gamma": 0...
latest_greeks JSON valide                               OK      keys = ['greeks', 'timestamp']
top-level keys = {timestamp, greeks}                    OK      actual = ['greeks', 'timestamp']
greeks keys = ['delta', 'gamma', 's

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `heartbeat absent` après seed surface | engine cycle skip → spot absent ou format invalide | `docker compose exec redis redis-cli GET latest_spot:EURUSD` ; vérifier que market-data tourne ET que le patch `_read_spot` est bien dans l'image (rebuild si pas sûr) |
| `latest_greeks JSON invalide` | bug serializer côté engine | inspecter manuellement avec `redis-cli GET latest_greeks:portfolio` |
| `greeks.delta = NaN` (avec positions=[]) | bug calcul (division par 0 implicite) | revoir `_aggregate_greeks` dans `engine.py` |
| `cohérence spot FAIL > 0.5%` | spot bouge trop vite OU bug pipeline | réessayer le notebook ; si persiste, comparer manuellement |
| `latest_pnl_curve présent mais inattendu` | quelqu'un a remplacé `_positions_stub` par du vrai fetch | mettre à jour le test §6 pour matcher le nouveau comportement |